# Screenplay QLoRA V8

## Install dependencies

In [0]:
%pip install --force-reinstall --no-deps "https://github.com/mlflow/mlflow/archive/e07bf239b9dcf95e23fce5e59dff091df17669bc.zip"

In [0]:
%pip install -U "bitsandbytes>=0.46.1"

In [0]:
%pip install -U "transformers==5.8.1" "trl==1.10.0"

In [0]:
%pip install -U "peft==0.19.1"

## Restart Python

In [0]:
dbutils.library.restartPython()

## Configure runtime

In [0]:
# Attach this notebook to Serverless GPU, 1xH100, AI v5 before running.
import os
import sys

os.environ["TQDM_DISABLE"] = "1"
os.environ["HF_HUB_DISABLE_PROGRESS_BARS"] = "1"
os.environ["MLFLOW_ENABLE_ARTIFACTS_PROGRESS_BAR"] = "false"

import tqdm.auto
import tqdm.std

tqdm.auto.tqdm = tqdm.std.tqdm
tqdm.auto.trange = tqdm.std.trange

## Import dependencies

In [0]:
import logging
import json
from contextlib import nullcontext
from pathlib import Path
from tempfile import mkdtemp

from drzavni_posao import MediaStore, MlflowTracker, Stage
from drzavni_posao.constants import ANONYMOUS_LABEL
from drzavni_posao.generation.dataset import (
    SFTDatasetBuilder,
    TranscriptQualityError,
    assign_folds,
    build_pair,
    select_splits,
)

import mlflow
import torch
import transformers
import trl
import peft
from mlflow.tracking import MlflowClient
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
    Qwen3_5ForConditionalGeneration,
    set_seed,
)
from trl import SFTConfig, SFTTrainer
from drzavni_posao.generation.prompt import build_episode_messages

## Configure logging

In [0]:
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s - %(name)s - %(levelname)s - %(message)s",
    datefmt="%Y-%m-%d %H:%M:%S",
    handlers=[logging.StreamHandler(sys.stdout)],
    force=True,
)
logging.getLogger("py4j").setLevel(logging.WARNING)
logging.getLogger("mlflow.system_metrics").setLevel(logging.WARNING)

## Configure data sources

In [0]:
SOURCE_EXPERIMENT_NAME = "/Users/luka.erceg@databricks.com/drzavni_posao/transcription"
SOURCE_RUN_ID = "6267d9cf1a5e45c79fb44ada55357711"
ORIGINAL_TRANSCRIPTION_RUN_ID = "1c519fba673841fa9188addf9a720ae6"
PREMISE_EXPERIMENT_NAME = (
    "/Users/luka.erceg@databricks.com/drzavni_posao/premise-generation"
)
PREMISE_RUN_ID = "4eadb1cda666464c90b6b01d71dbf574"
CORRECTION_RUN_ID = "57de21658fa5466e872ecf749accd121"
CORRECTION_CONFIDENCES = {"high"}
TRAINING_EXPERIMENT_NAME = (
    "/Users/luka.erceg@databricks.com/drzavni_posao/screenplay-qlora"
)
WORKSPACE_ROOT = Path("/Workspace/Users/luka.erceg@databricks.com/drzavni_posao")

## Configure model and LoRA

In [0]:
MODEL_KEY = "qwen38"  # Choose "qwen3", "qwen38", or "zora".
MODEL_NAMES = {
    "qwen3": "Qwen/Qwen3-8B",
    "qwen38": "Qwen/Qwen3.8-27B",
    "zora": "sovasoft/zora-v1.12",
}
if MODEL_KEY not in MODEL_NAMES:
    raise ValueError(f"Unknown model key {MODEL_KEY!r}; choose from {sorted(MODEL_NAMES)}")
MODEL_NAME = MODEL_NAMES[MODEL_KEY]
MODEL_CLASS = (
    Qwen3_5ForConditionalGeneration
    if MODEL_KEY == "qwen38"
    else AutoModelForCausalLM
)
LORA_TARGET_MODULES = (
    [
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj",
        "in_proj_qkv",
        "in_proj_z",
        "in_proj_b",
        "in_proj_a",
        "out_proj",
        "gate_proj",
        "up_proj",
        "down_proj",
    ]
    if MODEL_KEY == "qwen38"
    else ["q_proj", "v_proj"]
)

## Configure training

In [0]:
RUN_NAME = f"{MODEL_KEY}-qlora-full-episode-v8"
MAX_LENGTH = 8192
MAX_NEW_TOKENS = 2048
LORA_R = 16
LORA_ALPHA = 32
LORA_DROPOUT = 0.05
BATCH_SIZE = 1
GRADIENT_ACCUMULATION_STEPS = 8
LEARNING_RATE = 7.5e-5
TRAIN_EPOCHS = 2
WARMUP_RATIO = 0.05
MAX_GRAD_NORM = 0.3
SEED = 42

## Validate library versions

In [0]:
EXPECTED_TRANSFORMERS_VERSION = "5.8.1"
EXPECTED_TRL_VERSION = "1.10.0"
if transformers.__version__ != EXPECTED_TRANSFORMERS_VERSION:
    raise RuntimeError(
        f"Expected transformers {EXPECTED_TRANSFORMERS_VERSION}, "
        f"found {transformers.__version__}"
    )
if trl.__version__ != EXPECTED_TRL_VERSION:
    raise RuntimeError(f"Expected TRL {EXPECTED_TRL_VERSION}, found {trl.__version__}")

## Validate GPU runtime

In [0]:
if not torch.cuda.is_available():
    raise RuntimeError("This notebook requires a GPU")
GPU_NAME = torch.cuda.get_device_name(0)
if "H100" not in GPU_NAME:
    raise RuntimeError(f"Expected an H100, found {GPU_NAME}")

set_seed(SEED)
print(
    {
        "gpu": GPU_NAME,
        "model_key": MODEL_KEY,
        "model_name": MODEL_NAME,
        "transformers": transformers.__version__,
        "trl": trl.__version__,
        "peft": peft.__version__,
    }
)

## Load episode catalog

In [0]:
catalog_path = WORKSPACE_ROOT / "data/youtube_episodes.json"

## Load transcription run state

In [0]:
source_tracker = MlflowTracker(SOURCE_EXPERIMENT_NAME)
states = source_tracker.episode_run_states(SOURCE_RUN_ID)

## Load premises

In [0]:
mlflow.set_experiment(PREMISE_EXPERIMENT_NAME)
traces = mlflow.search_traces(
    run_id=PREMISE_RUN_ID,
    return_type="list",
)

premise_candidates = {}
duplicate_premise_keys = set()
empty_episode_keys = []
for trace in traces:
    if trace.info.tags.get("premise.complete") != "true":
        continue

    episode_key = trace.info.tags.get("episode_key")
    trace_metadata = trace.info.trace_metadata or {}
    state = states.get(episode_key)
    premise_source_run_id = trace_metadata.get("source_run_id")
    if state is None or premise_source_run_id not in {
        SOURCE_RUN_ID,
        ORIGINAL_TRANSCRIPTION_RUN_ID,
    }:
        continue
    outputs = trace.data.spans[0].outputs if trace.data.spans else None
    premise = ((outputs or {}).get("premise") or "").strip()
    if not episode_key or not premise:
        if episode_key:
            empty_episode_keys.append(episode_key)
        continue
    priority = (
        premise_source_run_id == SOURCE_RUN_ID
        and trace_metadata.get("source_episode_run_id") == state.run_id,
        trace.info.request_time or 0,
    )
    if episode_key in premise_candidates:
        duplicate_premise_keys.add(episode_key)
        if priority <= premise_candidates[episode_key][0]:
            continue
    premise_candidates[episode_key] = (priority, premise)

premises = {
    episode_key: candidate[1]
    for episode_key, candidate in premise_candidates.items()
}
print(f"matched premises: {len(premises)}")
print(f"duplicate premise keys resolved: {len(duplicate_premise_keys)}")

## Load corrections

In [0]:
corrections = {}
if CORRECTION_RUN_ID:
    correction_dir = Path(
        mlflow.artifacts.download_artifacts(
            run_id=CORRECTION_RUN_ID,
            artifact_path="corrections",
        )
    )
    for artifact in correction_dir.glob("*.json"):
        payload = json.loads(artifact.read_text())
        episode_key = payload.get("episode_key")
        edits = payload.get("edits")
        if not episode_key or not isinstance(edits, list):
            raise ValueError(f"Invalid correction artifact {artifact.name!r}")
        corrections[episode_key] = edits

## Define correction application

In [0]:
def apply_episode_corrections(dialogue, edits, confidences):
    lines = dialogue.splitlines()
    applied = 0
    skipped = 0
    for edit in edits:
        if edit.get("confidence") not in confidences:
            continue
        turn_id = edit.get("turn_id")
        original = edit.get("original")
        replacement = edit.get("replacement")
        if (
            type(turn_id) is not int
            or turn_id < 0
            or turn_id >= len(lines)
            or not isinstance(original, str)
            or not original
            or not isinstance(replacement, str)
            or not replacement
            or original == replacement
        ):
            skipped += 1
            continue

        prefix, separator, text = lines[turn_id].partition("] ")
        if not separator or text.count(original) != 1:
            skipped += 1
            continue
        lines[turn_id] = f"{prefix}{separator}{text.replace(original, replacement, 1)}"
        applied += 1

    return "\n".join(lines), applied, skipped

## Select episodes and folds

In [0]:
media = MediaStore(
    audio_dir=None,
    run_dir=Path("/tmp/screenplay-qlora"),
    catalog_path=catalog_path
)
source_episodes = [
    episode
    for episode in media.episodes()
    if episode.key in states
    and Stage.CHARACTER_ASSIGNER in states[episode.key].completed_stages
]
folds = assign_folds(source_episodes)
splits = select_splits(folds, test_fold=0, validation_fold=1)

## Load transcripts

In [0]:
episodes_with_premises = [
    episode for episode in source_episodes if episode.key in premises
]
anonymous_segments_removed = 0
for episode in episodes_with_premises:
    episode.transcript = source_tracker.stage_transcript(
        states[episode.key].run_id,
        Stage.CHARACTER_ASSIGNER,
    )
    anonymous_segments_removed += sum(
        segment.get("speaker") == ANONYMOUS_LABEL
        for segment in episode.transcript.segments
    )

## Build corrected training pairs

In [0]:
rows = []
quality_exclusions = {}
correction_counts = {}
correction_edits_applied = 0
correction_edits_skipped = 0
for episode in episodes_with_premises:
    premise = premises[episode.key]
    try:
        pair = build_pair(episode, premise)
    except TranscriptQualityError as error:
        quality_exclusions[episode.key] = error.reasons
        continue
    pair["dialogue"], applied, skipped = apply_episode_corrections(
        pair["dialogue"],
        corrections.get(episode.key, []),
        CORRECTION_CONFIDENCES,
    )
    if applied or skipped:
        correction_counts[episode.key] = {
            "applied": applied,
            "skipped": skipped,
        }
    correction_edits_applied += applied
    correction_edits_skipped += skipped
    rows.append({**pair, "split": splits[episode.key]})

## Review dataset filtering

In [0]:
if not rows:
    raise ValueError("No valid premise/dialogue pairs found")

print(f"valid pairs: {len(rows)}")
print(f"empty premises filtered: {len(empty_episode_keys)}")
print(f"quality exclusions: {len(quality_exclusions)}")
print(f"anonymous segments removed: {anonymous_segments_removed}")
print(f"correction edits applied: {correction_edits_applied}")
print(f"correction edits skipped: {correction_edits_skipped}")
if quality_exclusions:
    print(quality_exclusions)
print(
    "splits:",
    {
        split: sum(row["split"] == split for row in rows)
        for split in ("train", "validation", "test")
    },
)

## Load tokenizer

In [0]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

## Build tokenized dataset

In [0]:
dataset_builder = SFTDatasetBuilder(
    tokenizer,
    max_length=MAX_LENGTH,
    enable_thinking=False,
)
split_rows, overlong_episode_keys = dataset_builder.prepare_rows(rows)
dataset = dataset_builder.build_dataset(split_rows)
training_rows = [
    row for split in ("train", "validation", "test") for row in split_rows[split]
]

print(dataset)
print(f"overlong episodes filtered: {overlong_episode_keys}")
print(
    "token range:",
    min(row["token_count"] for row in training_rows),
    max(row["token_count"] for row in training_rows),
)

## Load quantized model

In [0]:
quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)
model = MODEL_CLASS.from_pretrained(
    MODEL_NAME,
    quantization_config=quantization_config,
    torch_dtype=torch.bfloat16,
    device_map={"": 0},
)
model.config.use_cache = False
if hasattr(model.config, "text_config"):
    model.config.text_config.use_cache = False

## Attach LoRA adapters

In [0]:
available_module_names = {
    name.rsplit(".", 1)[-1] for name, _ in model.named_modules()
}
missing_target_modules = sorted(set(LORA_TARGET_MODULES) - available_module_names)
if missing_target_modules:
    raise RuntimeError(f"Missing LoRA target modules: {missing_target_modules}")

model = prepare_model_for_kbit_training(
    model,
    use_gradient_checkpointing=True,
)
model = get_peft_model(
    model,
    LoraConfig(
        r=LORA_R,
        lora_alpha=LORA_ALPHA,
        lora_dropout=LORA_DROPOUT,
        target_modules=LORA_TARGET_MODULES,
        bias="none",
        task_type="CAUSAL_LM",
    ),
)
if MODEL_KEY == "qwen38":
    trainable_vision_parameters = [
        name
        for name, parameter in model.named_parameters()
        if parameter.requires_grad and ("vision" in name or "visual" in name)
    ]
    if trainable_vision_parameters:
        raise RuntimeError(
            f"Vision parameters unexpectedly trainable: {trainable_vision_parameters[:5]}"
        )
model.print_trainable_parameters()
print(
    "GPU memory after model preparation (GiB):",
    round(torch.cuda.max_memory_allocated() / 1024**3, 2),
)

## Configure trainer

In [0]:
OUTPUT_DIR = mkdtemp(prefix=f"{MODEL_KEY}-qlora-")
training_args = SFTConfig(
    output_dir=OUTPUT_DIR,
    max_length=MAX_LENGTH,
    completion_only_loss=True,
    packing=False,
    num_train_epochs=TRAIN_EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRADIENT_ACCUMULATION_STEPS,
    learning_rate=LEARNING_RATE,
    lr_scheduler_type="cosine",
    warmup_ratio=WARMUP_RATIO,
    weight_decay=0.01,
    max_grad_norm=MAX_GRAD_NORM,
    optim="paged_adamw_8bit",
    bf16=True,
    tf32=True,
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": False},
    eval_strategy="epoch",
    eval_on_start=True,
    save_strategy="epoch",
    save_total_limit=1,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    logging_steps=1,
    report_to="mlflow",
    run_name=RUN_NAME,
    seed=SEED,
    data_seed=SEED,
)
trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=dataset["train"],
    eval_dataset=dataset["validation"],
    processing_class=tokenizer,
)

## Validate completion-only masking

In [0]:
first_training_row = split_rows["train"][0]
expected_prompt_ids = tokenizer.apply_chat_template(
    first_training_row["prompt"],
    tokenize=True,
    add_generation_prompt=True,
    enable_thinking=False,
    return_dict=True,
)["input_ids"]
expected_full_ids = tokenizer.apply_chat_template(
    first_training_row["prompt"] + first_training_row["completion"],
    tokenize=True,
    add_generation_prompt=False,
    enable_thinking=False,
    return_dict=True,
)["input_ids"]
if expected_full_ids[: len(expected_prompt_ids)] != expected_prompt_ids:
    raise RuntimeError("Tokenizer prompt mismatch before completion")

first_training_example = trainer.train_dataset[0]
input_ids = first_training_example["input_ids"]
labels = first_training_example.get("labels")
if labels is None:
    raise RuntimeError("TRL did not create completion-only labels")
masked_prompt_tokens = sum(label == -100 for label in labels)
supervised_completion_tokens = len(labels) - masked_prompt_tokens
if input_ids != expected_full_ids:
    raise RuntimeError("TRL tokenization differs from the expected chat template")
if masked_prompt_tokens != len(expected_prompt_ids):
    raise RuntimeError(
        f"Expected {len(expected_prompt_ids)} masked prompt tokens, "
        f"found {masked_prompt_tokens}"
    )
if labels[:masked_prompt_tokens] != [-100] * masked_prompt_tokens:
    raise RuntimeError("Prompt labels are not fully masked")
if labels[masked_prompt_tokens:] != input_ids[masked_prompt_tokens:]:
    raise RuntimeError("Completion labels do not match completion tokens")
print(
    "completion-only labels:",
    {
        "prompt_tokens": masked_prompt_tokens,
        "completion_tokens": supervised_completion_tokens,
    },
)

## Prepare MLflow run metadata

In [0]:
run_tags = {
    "model_key": MODEL_KEY,
    "model_name": MODEL_NAME,
    "training_method": "qlora",
    "dataset_paradigm": "full_episode",
    "source_run_id": SOURCE_RUN_ID,
    "premise_run_id": PREMISE_RUN_ID,
    "correction_run_id": CORRECTION_RUN_ID or "none",
}
run_parameters = {
    "model_key": MODEL_KEY,
    "max_length": MAX_LENGTH,
    "lora_r": LORA_R,
    "lora_alpha": LORA_ALPHA,
    "lora_dropout": LORA_DROPOUT,
    "lora_target_modules": ",".join(LORA_TARGET_MODULES),
    "batch_size": BATCH_SIZE,
    "gradient_accumulation_steps": GRADIENT_ACCUMULATION_STEPS,
    "learning_rate": LEARNING_RATE,
    "epochs": TRAIN_EPOCHS,
    "warmup_ratio": WARMUP_RATIO,
    "max_grad_norm": MAX_GRAD_NORM,
    "correction_run_id": CORRECTION_RUN_ID or "none",
    "correction_confidences": ",".join(sorted(CORRECTION_CONFIDENCES)),
    "transformers_version": transformers.__version__,
    "trl_version": trl.__version__,
    "peft_version": peft.__version__,
    "seed": SEED,
}

## Prepare dataset manifest

In [0]:
dataset_metrics = {
    "dataset.valid_pairs": len(rows),
    "dataset.empty_premises": len(empty_episode_keys),
    "dataset.quality_exclusions": len(quality_exclusions),
    "dataset.anonymous_segments_removed": anonymous_segments_removed,
    "dataset.overlong_examples": len(overlong_episode_keys),
    "dataset.correction_edits_applied": correction_edits_applied,
    "dataset.correction_edits_skipped": correction_edits_skipped,
    "dataset.corrected_episodes": sum(
        counts["applied"] > 0 for counts in correction_counts.values()
    ),
    **{
        f"dataset.{split}_examples": len(items)
        for split, items in split_rows.items()
    },
}
dataset_manifest = {
    "source_experiment_name": SOURCE_EXPERIMENT_NAME,
    "source_run_id": SOURCE_RUN_ID,
    "premise_experiment_name": PREMISE_EXPERIMENT_NAME,
    "premise_run_id": PREMISE_RUN_ID,
    "correction_run_id": CORRECTION_RUN_ID,
    "correction_confidences": sorted(CORRECTION_CONFIDENCES),
    "correction_counts": correction_counts,
    "empty_episode_keys": empty_episode_keys,
    "quality_exclusions": quality_exclusions,
    "anonymous_segments_removed": anonymous_segments_removed,
    "overlong_episode_keys": overlong_episode_keys,
    "rows": [
        {
            "episode_key": row["episode_key"],
            "split": row["split"],
            "premise": row["premise"],
            "token_count": row["token_count"],
        }
        for row in training_rows
    ],
}

## Train and save adapter

In [0]:
mlflow.set_experiment(TRAINING_EXPERIMENT_NAME)
with mlflow.start_run(run_name=RUN_NAME) as run:
    TRAINING_RUN_ID = run.info.run_id
    mlflow.set_tags(run_tags)
    mlflow.log_params(run_parameters)
    mlflow.log_metrics(dataset_metrics)
    mlflow.log_dict(dataset_manifest, "dataset_manifest.json")

    train_result = trainer.train()
    eval_metrics = trainer.evaluate()
    mlflow.log_metrics(
        {
            key: float(value)
            for key, value in eval_metrics.items()
            if isinstance(value, (int, float))
        }
    )

    trainer.save_model(OUTPUT_DIR)
    tokenizer.save_pretrained(OUTPUT_DIR)
    mlflow.log_artifacts(OUTPUT_DIR, artifact_path="adapter")

print(f"training run: {TRAINING_RUN_ID}")
print(eval_metrics)

## Configure held-out generation

In [0]:
model.gradient_checkpointing_disable()
model.config.use_cache = True
if hasattr(model.config, "text_config"):
    model.config.text_config.use_cache = True
model.eval()


def generate(premise, adapter_enabled):
    messages = build_episode_messages(premise)
    inputs = tokenizer.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=True,
        enable_thinking=False,
        return_tensors="pt",
        return_dict=True,
    )
    inputs = {key: value.to(model.device) for key, value in inputs.items()}
    adapter_context = nullcontext() if adapter_enabled else model.disable_adapter()

    set_seed(SEED)
    with (
        adapter_context,
        torch.inference_mode(),
        torch.autocast("cuda", dtype=torch.bfloat16),
    ):
        output = model.generate(
            **inputs,
            do_sample=True,
            temperature=0.8,
            top_p=0.95,
            top_k=0,
            repetition_penalty=1.15,
            max_new_tokens=MAX_NEW_TOKENS,
            eos_token_id=tokenizer.eos_token_id,
            pad_token_id=tokenizer.eos_token_id,
        )

    generated = output[0, inputs["input_ids"].shape[1] :]
    return tokenizer.decode(generated, skip_special_tokens=True).strip()

## Generate representative comparison

In [0]:
heldout = split_rows["test"][8]
comparison = {
    "episode_key": heldout["episode_key"],
    "premise": heldout["premise"],
    "base": generate(heldout["premise"], adapter_enabled=False),
    "qlora": generate(heldout["premise"], adapter_enabled=True),
}
MlflowClient().log_dict(
    TRAINING_RUN_ID,
    comparison,
    "heldout_comparison.json",
)
print(comparison)

## Generate all held-out comparisons (optional)

In [0]:
HOLDOUT_MAX_NEW_TOKENS = 1024
holdout_rows = split_rows["test"]
if not holdout_rows:
    raise RuntimeError("No holdout episodes found")

MAX_NEW_TOKENS = HOLDOUT_MAX_NEW_TOKENS
client = MlflowClient()

for index, heldout in enumerate(holdout_rows, start=1):
    episode_key = heldout["episode_key"]
    print(f"[{index}/{len(holdout_rows)}] generating episode {episode_key}")

    comparison = {
        "episode_key": episode_key,
        "holdout_index": index,
        "premise": heldout["premise"],
        "reference": heldout["completion"][0]["content"],
        "base": generate(heldout["premise"], adapter_enabled=False),
        "qlora": generate(heldout["premise"], adapter_enabled=True),
        "generation_config": {
            "seed": SEED,
            "max_new_tokens": HOLDOUT_MAX_NEW_TOKENS,
            "do_sample": True,
            "temperature": 0.8,
            "top_p": 0.95,
            "top_k": 0,
            "repetition_penalty": 1.15,
        },
    }

    artifact_path = f"heldout-comparisons/episode-{index:03d}.json"
    client.log_dict(TRAINING_RUN_ID, comparison, artifact_path)
    print(f"saved {artifact_path}")

print(
    f"Saved {len(holdout_rows)} comparisons under "
    f"runs:/{TRAINING_RUN_ID}/heldout-comparisons/"
)